# Train a language model on Tiny Shakespeare

You will train a decoder-only transformer with the parts current open models use: RMSNorm, rotary positions, grouped-query attention and a gated MLP. The corpus is Tiny Shakespeare, about 1.1 MB of plays. By the end the model writes Shakespeare-flavoured text from a prompt, and you will have restored a checkpoint from disk and sampled from it.

The notebook uses Dew's public API end to end. The data pipeline packs the corpus into fixed windows, `LMObjective` owns the loss, and `ObjectiveTrainer` owns gradients, the EMA, sharding, checkpoints and logging. This is the same code a scaled-up run uses, not a hand-written loop.

**Expected time.** About 15 minutes on an A100, 15 to 20 on a TPU v5e, 25 to 35 on an RTX 4080. The first epoch includes a few minutes of XLA compilation that the compilation cache skips on later runs. A CPU run takes days; do not run this configuration there.

In [ ]:
# Install cell: only runs on Colab (import google.colab succeeds there).
# A TPU runtime gets jax[tpu] instead of jax[cuda12]. Locally this cell does nothing.
import os
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB:
    # The backend is decided by the accelerator Colab attached; jax is not
    # imported yet, so this looks at what the runtime provides.
    jax_spec = "jax[tpu]" if "COLAB_TPU_ADDR" in os.environ else "jax[cuda12]"
    %pip install -q "dew-ml @ git+https://github.com/AshishKumar4/dew" {jax_spec}

In [ ]:
# Every size knob in one place.
SEQUENCE_LENGTH = 256   # tokens of context per example
BATCH_SIZE = 64         # sequences per step, split across all devices
EPOCHS = 60             # 66 steps per epoch on this corpus, about 4000 steps total
LEARNING_RATE = 1e-3    # peak, after 200 warmup steps, cosine decay to a tenth
EMB_FEATURES = 512      # model width
NUM_LAYERS = 12
NUM_HEADS = 8
MAX_NEW_TOKENS = 400    # per sample; the KV cache is sized for the training context or this, whichever is longer
PROMPT = "ROMEO:"
TOKENIZER = "byte"      # "byte" or "gpt2"; see the tokenizer cell
DATA_DIR = "shakespeare"

In [ ]:
import jax
print("devices:", jax.devices())
print("backend:", jax.default_backend())

## The data

A language model predicts the next token, so the corpus becomes one flat stream of token ids and the loader cuts it into windows of `SEQUENCE_LENGTH + 1`. The first 256 ids of a row are the inputs, the last 256 are the same ids shifted by one, which makes the targets; nothing is shifted twice. Each window overlaps the previous one by exactly one token, and the loader yields `{"text": int32[B, 257]}`.

The tokenizer is a switch. **Bytes** give a vocabulary of 256 and need no download, at the price of several tokens per common word. **GPT-2 BPE** needs a 50257-row embedding table, which dominates a small model's parameter count, but packs the same text into about a quarter of the tokens, so each step sees more words. The cell below writes the same `train.bin`, `val.bin` and `meta.json` that `tools/tokenize_text.py` produces in a repo checkout, and the model is always built for the vocabulary the files were written with.

In [ ]:
import json
import urllib.request
from pathlib import Path

import numpy as np
from dew.data.text import ByteTokenizer, HFTokenizer

DATA_DIR = Path(DATA_DIR)
DATA_DIR.mkdir(parents=True, exist_ok=True)
raw = DATA_DIR / "shakespeare.txt"
if not raw.exists():
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
        raw)
print("raw text:", raw.stat().st_size, "bytes")

tokenizer = ByteTokenizer() if TOKENIZER == "byte" else HFTokenizer(TOKENIZER)
ids = np.asarray(tokenizer.encode(raw.read_text(encoding="utf-8")))
val_len = int(round(len(ids) * 0.02))
val, train = ids[:val_len], ids[val_len:]
# uint8 covers the byte vocabulary, uint16 every HF one; the loader reads
# the dtype back from meta.json. The head of the stream is validation, as
# tools/tokenize_text.py splits it.
dtype = np.dtype("uint8") if tokenizer.vocab_size <= 256 else np.dtype("uint16")
val.astype(dtype).tofile(DATA_DIR / "val.bin")
train.astype(dtype).tofile(DATA_DIR / "train.bin")
meta = {"tokenizer": TOKENIZER, "vocab_size": tokenizer.vocab_size, "dtype": dtype.name,
        "train_tokens": len(train), "val_tokens": len(val)}
(DATA_DIR / "meta.json").write_text(json.dumps(meta, indent=2))
print(meta)

In [ ]:
import numpy as np
from dew.data.dataloaders import get_token_dataset_grain
from dew.data.sources.text import TokenFileSource

data = get_token_dataset_grain(
    f"{DATA_DIR}/train.bin", f"{DATA_DIR}/val.bin",
    batch_size=BATCH_SIZE, seq_len=SEQUENCE_LENGTH, worker_count=2)
print("train windows:", data["train_len"], "val windows:", data["val_len"])

batch = next(iter(data["train"]()))
print(batch["text"].shape, batch["text"].dtype)

# TokenFileSource is what the loader reads through: a memory-mapped view over
# the token file where record i is the window starting at i * seq_len. The
# validation windows scored at the end of the notebook are taken from it
# directly, in this process, so the scoring path needs no second loader.
val_source = TokenFileSource(f"{DATA_DIR}/val.bin", SEQUENCE_LENGTH)
val_batches = [
    np.stack([val_source[i]["text"]
              for i in range(start, min(start + BATCH_SIZE, len(val_source)))])
    for start in range(0, len(val_source), BATCH_SIZE)
]
print("validation windows held for scoring:", len(val_source))

## The model and the objective

`build_model("causal_transformer", ...)` constructs the decoder from the registry. `dtype="bfloat16"` is the compute dtype; the parameters stay fp32. `attention_impl="auto"` picks the fused attention kernel the hardware has. The vocabulary comes from `meta.json`, and `max_seq_len` has to cover the training context plus everything the final sample generates, because the KV cache is allocated once at that length.

`LMObjective` is next-token cross entropy. It shifts the batch itself, casts the logits to fp32 before the softmax (a bf16 logsumexp over a large vocabulary loses enough precision to move the loss and the gradient), and returns `ce`, `perplexity` and `token_accuracy` beside the loss, which the trainer logs as `train/<name>`. At validation it reads the EMA weights, reports teacher-forced cross entropy on held-out windows, and writes a sample of text from a fixed prompt.

In [ ]:
import optax
from dew.data.text import ByteTokenizer, HFTokenizer
from dew.objectives.lm import LMObjective
from dew.registry import apply_precision_policy, build_model

tokenizer = ByteTokenizer() if TOKENIZER == "byte" else HFTokenizer(TOKENIZER)

model_config = apply_precision_policy("causal_transformer", dict(
    vocab_size=meta["vocab_size"], emb_features=EMB_FEATURES,
    num_layers=NUM_LAYERS, num_heads=NUM_HEADS,
    max_seq_len=max(SEQUENCE_LENGTH, len(tokenizer.encode(PROMPT)) + MAX_NEW_TOKENS),
), dtype="bfloat16", attention_impl="auto")
model = build_model("causal_transformer", model_config)

objective = LMObjective(
    model, SEQUENCE_LENGTH, vocab_size=meta["vocab_size"],
    samples={
        "prompt": tokenizer.encode(PROMPT),
        "max_new_tokens": 200,
        "temperature": 0.8,
        "top_k": 40,
        "decode": tokenizer.decode,
    })

params = jax.eval_shape(objective.init_params, jax.random.PRNGKey(0))
print(f"{sum(x.size for x in jax.tree_util.tree_leaves(params)) / 1e6:.1f}M parameters")

## Training

`ObjectiveTrainer` compiles the objective's loss into one jitted step with the optimizer update and the EMA average inside it, shards it over every device, and writes checkpoints under `./checkpoints/shakespeare`. With `eval_metrics=[get_perplexity_metric()]` every epoch also reports `val/perplexity` on the held-out split. One epoch here is 66 steps.

In [ ]:
from dew.eval import get_perplexity_metric
from dew.training import ObjectiveTrainer

TOTAL_STEPS = EPOCHS * (data["train_len"] // BATCH_SIZE)
schedule = optax.warmup_cosine_decay_schedule(
    init_value=LEARNING_RATE * 0.01, peak_value=LEARNING_RATE,
    warmup_steps=200, decay_steps=TOTAL_STEPS, end_value=LEARNING_RATE * 0.1)

trainer = ObjectiveTrainer(
    model, optax.adamw(schedule), objective=objective, input_config=None,
    rngs=jax.random.PRNGKey(0), name="shakespeare",
    eval_metrics=[get_perplexity_metric()],
    checkpoint_base_path="./checkpoints")
state = trainer.fit(data, training_steps_per_epoch=data["train_len"] // BATCH_SIZE,
                    epochs=EPOCHS, val_steps_per_epoch=8)

## Perplexity

Perplexity is `exp(cross entropy)`: on average, how many tokens the model was choosing between at each position. A byte-level model that has just started scores around 100, a trained one lands near 3 on Shakespeare. The number the trainer printed at each epoch came from the EMA weights on held-out windows. The cell below recomputes it over every validation window, through the same objective method the validation step uses.

In [ ]:
teacher_forced_ce = jax.jit(
    lambda params, tokens: objective.shifted_cross_entropy(params, tokens)[0])
ces = [float(teacher_forced_ce(state.ema_params, tokens)) for tokens in val_batches]
val_ce = float(np.mean(ces))
print(f"validation cross entropy {val_ce:.4f}  perplexity {np.exp(val_ce):.2f}")

## Generation and the KV cache

A decoder generates one token at a time, and each new token attends to every token before it. Recomputing all of that per step is quadratic work. The KV cache removes it: attention keys and values for every position live in a buffer sized `max_seq_len`, the prompt runs through the model once to fill the cache (the prefill), and every later step runs a single token through the model, appends its keys and values, and reads the logits for the next token. The rotary positions come from the cache index rather than the row index of the token, so a token gets the same position vector whether it arrived in the prefill or comes back later as a decode step.

`generate` does the prefill and then the whole decode loop in one `lax.scan`, so a generation is one compiled program rather than `max_new_tokens` compiled steps. `temperature` flattens or sharpens the distribution, with 0 meaning greedy: always the most likely token. `top_k` restricts each step to the k most likely tokens before sampling, which keeps a small model from wandering into rare garbage.

In [ ]:
import jax.numpy as jnp
from dew.sampling.text import generate

prompt = jnp.asarray([tokenizer.encode(PROMPT)], jnp.int32)

greedy = generate(model, state.ema_params, prompt, max_new_tokens=MAX_NEW_TOKENS,
                  rng=jax.random.PRNGKey(0), temperature=0.0)
print("greedy:\n" + tokenizer.decode(greedy[0]) + "\n")

sampled = generate(model, state.ema_params, prompt, max_new_tokens=MAX_NEW_TOKENS,
                   rng=jax.random.PRNGKey(1), temperature=0.8, top_k=40)
print("temperature=0.8, top_k=40:\n" + tokenizer.decode(sampled[0]))

## Checkpoints

`fit` writes a checkpoint at the end of the run, and at any epoch that improved on the best, under `./checkpoints/shakespeare`. The checkpoint holds the parameters, the EMA, the optimizer state, the step and the data position, so a resumed run continues from the next unseen batch. `load_from_checkpoint` restores a state onto whatever devices the process has, and a checkpoint written on one mesh loads onto another.

In [ ]:
import numpy as np
from dew.sampling.loading import load_from_checkpoint

trainer.wait_for_checkpoints()
# load_from_checkpoint restores the raw state dict the trainer saved:
# params, ema_params, the optimizer state and the step.
restored = load_from_checkpoint("./checkpoints/shakespeare", step="latest")
print("restored at step", int(np.asarray(restored["step"])))
tokens = generate(model, restored["ema_params"], prompt, max_new_tokens=120,
                  rng=jax.random.PRNGKey(0), temperature=0.8, top_k=40)
print(tokenizer.decode(tokens[0])[:400])

## Where to go next

Set `TOKENIZER = "gpt2"` in the parameters cell and rerun everything: the embedding table grows to 50257 rows, the corpus packs into about a quarter of the tokens, and the samples read noticeably better after the same number of steps. The same run from the command line is `python recipes/lm/train.py --data.dataset shakespeare --sequence-length 256 --model.config '{"emb_features": 512, "num_layers": 12, "num_heads": 8}'`. The next notebook in this series trains the same objective on eight devices at once.